# Fitting a linear model with a genetic algorithm

How can a population of guesses evolve into a useful set of regression coefficients? In this notebook, we generate data from a known model, fit it directly with least squares, and use that solution as a benchmark for a genetic algorithm (GA).

Unlike the unbounded objective in `genetic_algorithm.ipynb`, this problem has a meaningful target: minimize prediction error. A GA is not needed to solve ordinary linear least squares; it is used here to make the mechanics of population-based search visible.

## Learning goals

By the end, you should be able to:

- Translate a coefficient vector into predictions using matrix multiplication.
- Interpret mean squared error and the coefficient of determination, $R^2$.
- Trace selection, crossover, mutation, and replacement through one generation.
- Compare a stochastic search with a direct least-squares solution.
- Identify what the stopping rule and training score do—and do not—establish.

## Before you run

Use Python 3 with NumPy and `tqdm` installed. All GA functions are defined in this notebook; this example does not import `ga.py`. Run the cells in order. Several functions depend on variables defined later in the main-program cell, so running isolated cells with old kernel state can produce misleading results.

The default run evaluates 1,000 candidates per generation on 1,000 observations and prints a progress bar for every generation. For a first walkthrough, reduce `max_generations` to a small value such as `5`; this is an iteration limit, not a guarantee of a good fit.

Randomness comes from Python's `random` module. For repeatable experiments in the same environment, add `import random as rng` and `rng.seed(42)` before generating data and populations, then rerun cells in the same order. `numpy.random.seed` alone does not control this notebook's random draws.

## 1. Generate data with a known answer

Each observation has four features sampled independently from `[0, 1)`. The target is generated by

$$y_i=0.4x_{i1}-0.3x_{i2}+0.2x_{i3}-0.1x_{i4}.$$

The true coefficient vector is $\boldsymbol{\beta}_{\mathrm{true}}=(0.4,-0.3,0.2,-0.1)$. There is **no added noise and no intercept**. For example, an observation $(0.5,0.2,0.8,0.1)$ has target $0.29$.

For $n=1000$, `inputs` has shape `(1000, 4)` and `outputs` has shape `(1000,)`: the targets are a one-dimensional array, not a column matrix. A candidate coefficient vector has shape `(4,)`, and its predictions are

$$\hat{\mathbf y}=X\boldsymbol{\beta}.$$

Defining `generate_data` does not yet create the dataset; the function is called in the benchmark and main-program cells. Those calls generate separate datasets.

**Pause and predict:** Two true coefficients are negative. Could a search initialized with only nonnegative coefficients recover them using crossover alone?

In [37]:
from random import random, sample, choice
from math import floor
from tqdm import tqdm
from numpy import array, dot, mean
from numpy.linalg import pinv

In [38]:
# Generate data - 1000 data points with 4 independent variables
#
# x will be a 1000 x 4 matrix
# y will be a one-dimensional array of length 1000

def generate_data(num_data_points=1000):
    coeff = [0.4, -0.3, 0.2, -0.1]
    x = [[random() for j in range(len(coeff))] for i in range(num_data_points)]
    y = [dot(i, coeff) for i in x]
    return array(x), array(y)

## 2. Define success and compute a benchmark

For predictions $\hat y_i$, define the residual $r_i=y_i-\hat y_i$. The residual sum of squares and mean squared error are

$$\mathrm{SSE}=\sum_{i=1}^{n}(y_i-\hat y_i)^2,
\qquad \mathrm{MSE}=\frac{\mathrm{SSE}}{n}.$$

The code calls the residual sum `SSR` and returns MSE under the key `error`. This is a **mean squared** error, not a mean signed error or a root mean squared error. Smaller is better, with zero representing exact predictions.

The total variation about the observed mean is

$$\mathrm{SST}=\sum_{i=1}^{n}(y_i-\bar y)^2,$$

and the coefficient of determination is

$$R^2=1-\frac{\mathrm{SSE}}{\mathrm{SST}}.$$

The code reports `COD = 100 * R²`. Thus `COD = 99.9` means $R^2=0.999$, not that 99.9% of individual predictions are correct. A score of 100 corresponds to an exact fit; zero means the same squared error as always predicting $\bar y$; a negative value means worse error than that baseline. The formula is undefined when all targets are identical ($\mathrm{SST}=0$), a case the code does not handle.

For a fixed dataset with positive SST, minimizing MSE and maximizing COD rank candidates identically.

### The direct least-squares reference

`multiple_linear_regression` computes

$$\hat{\boldsymbol{\beta}}=(X^TX)^+X^T\mathbf y,$$

where $+$ denotes the pseudoinverse (`pinv`). It returns a dictionary containing `coeff`, `error`, and `COD`.

For this noiseless dataset and a full-column-rank design matrix, least squares recovers the generating coefficients up to floating-point precision. The saved MSE of about $10^{-31}$ is numerical roundoff, not evidence of meaningful prediction error. This gives the GA a concrete reference: coefficients near $(0.4,-0.3,0.2,-0.1)$ and MSE near zero.

The implementation forms $X^TX$ explicitly. For more demanding numerical work, a direct least-squares solver such as `numpy.linalg.lstsq(X, y, rcond=None)` avoids forming the normal equations, which can worsen conditioning. The current function is retained here as the teaching benchmark.

In [39]:
# inputs is a 1000 x 4 matrix
# outputs is a one-dimensional array of length 1000
#
# These are the inputs and outputs generated in the previous step

def multiple_linear_regression(inputs, outputs):
    X, Y = array(inputs), array(outputs)
    X_t, Y_t = X.transpose(), Y.transpose()
    coeff = dot((pinv((dot(X_t, X)))), (dot(X_t, Y)))
    Y_p = dot(X, coeff)
    Y_mean = mean(Y)
    SST = array([(i - Y_mean) ** 2 for i in Y]).sum()
    SSR = array([(i - j) ** 2 for i, j in zip(Y, Y_p)]).sum()
    COD = (1 - (SSR / SST)) * 100.0
    av_error = (SSR / len(Y))
    return {'COD': COD, 'coeff': coeff, 'error': av_error}

# Test this function with the generated data

inputs, outputs = generate_data()
print(multiple_linear_regression(inputs, outputs))


{'COD': 100.0, 'coeff': array([ 0.4, -0.3,  0.2, -0.1]), 'error': 1.1712154761536559e-31}


## 3. Specify a stopping rule

The intended rule is to stop when the candidate supplied to `check_termination_condition` reaches **COD ≥ 99.9**, or when the generation counter equals `max_generations`.

The score threshold is equivalent to $\mathrm{SSE}/\mathrm{SST}\leq0.001$. It measures fit on the generated training data; it does not certify that the coefficients are exact or that the model will generalize to new observations.

**Read the loop carefully:** the main program supplies the previous stored winner to this function, rather than the winner just evaluated. On the first iteration, it supplies a separate random candidate. This introduces a one-generation delay. The counter also starts at zero and is checked after evaluation, so `max_generations = 100` allows evaluations labeled 0 through 100: up to **101 population evaluations**. These existing behaviors are explained again beside the main loop.

In [40]:
def check_termination_condition(best_individual):
    if ((best_individual['COD'] >= 99.9)
            or (generation_count == max_generations)):
        return True
    else:
        return False

## 4. Represent an individual

An **individual**, or chromosome, is a list of four trial coefficients. Each coefficient is a **gene**. For example, `[0.1, 0.2, 0.3, 0.4]` represents

$$\hat y=0.1x_1+0.2x_2+0.3x_3+0.4x_4.$$

`create_individual` draws each coefficient from `[0, 1)`. This is only an initialization range; coefficients are not constrained to that interval later.

Because two true coefficients are negative, initialization cannot already contain the exact answer. Crossover only copies existing values, so **mutation is essential** to introduce negative coefficients. The small demonstration below creates one candidate without evaluating its quality.

In [41]:
def create_individual(individual_size):
    return [random() for i in range(individual_size)]

# Test this function

individual_size_test = 4
print(create_individual(individual_size_test))


[0.726559972858926, 0.8489435008408955, 0.18333288175468854, 0.2650957246691381]


## 5. Build a population of guesses

`create_population` returns a list of independent coefficient lists. The demonstration creates three individuals so that their structure is easy to inspect; the main run uses 1,000 individuals, each with four genes.

Do not confuse the **number of observations** with the **population size**. Both happen to be 1,000 by default, but they play different roles: every candidate in the population is evaluated against every observation in the dataset.

In [42]:
def create_population(individual_size, population_size):
    return [create_individual(individual_size) for i in range(population_size)]

# test this function

individual_size_test = 4
population_size_test = 3
print(create_population(individual_size_test, population_size_test))

[[0.336150347687028, 0.8296016633148237, 0.02217210111188639, 0.6891906025729413], [0.7105531358572809, 0.8398830637808755, 0.6665039054700508, 0.06527176757619568], [0.40814863410501323, 0.6146147031944521, 0.7909646137832319, 0.733381038511307]]


## 6. Score one candidate

`get_fitness` computes predictions with `dot(inputs, individual)` and evaluates them against the target array. It returns:

| Key | Meaning | Preferred direction |
| --- | --- | --- |
| `coeff` | The candidate's four coefficients | Compare with the known generating vector |
| `error` | Mean squared prediction error | Smaller |
| `COD` | $100R^2$ on the current dataset | Larger |

Unlike the least-squares function, this function does **not** solve for coefficients. It only scores the coefficients it is given.

The target array `outputs` is a **global variable**, not an argument. Whenever you change `inputs`, ensure `outputs` comes from the same dataset in the same observation order. A stale target array can invalidate the comparison even if the shapes match.

In [43]:
def get_fitness(individual, inputs):
    predicted_outputs = dot(array(inputs), array(individual))
    output_mean = mean(outputs)
    SST = array(
        [(i - output_mean) ** 2 for i in outputs]
        ).sum()
    SSR = array(
        [(i - j) ** 2 for i, j in zip(outputs, predicted_outputs)]
        ).sum()
    COD = (1 - (SSR / SST)) * 100.0
    average_error = (SSR / len(outputs))
    return {'COD': COD, 'error': average_error, 'coeff': individual}


## 7. Select parents from the current population

`evaluate_population` scores every individual, sorts the results by increasing MSE, and retains the first `selection_size` entries. With the defaults, this means the best **100 of 1,000** candidates become eligible parents.

Selection here is deterministic: lower error earns a place in the parent pool. The later choice of mating pairs is random within that pool. The `tqdm` bar reports progress through candidate evaluations, not progress toward an optimal fit.

The function also appends the current generation's winning coefficient vector to `best_individuals_stash` and prints its MSE and COD. Despite its name, this stash is a **history of generation winners**, not a running best-so-far archive. The newest entry need not be the best entry ever seen, and it is not automatically reinserted into the population.

This function depends on the global variables `inputs`, `selection_size`, and `best_individuals_stash`; the main-program cell initializes them before calling it.

In [44]:
# tqdm is used to display a progress bar

def evaluate_population(population):
    fitness_list = [get_fitness(individual, inputs)
                    for individual in tqdm(population)]
    error_list = sorted(fitness_list, key=lambda i: i['error'])
    best_individuals = error_list[: selection_size]
    best_individuals_stash.append(best_individuals[0]['coeff'])
    print('Error: ', best_individuals[0]['error'],
          'COD: ', best_individuals[0]['COD'])
    return best_individuals

## 8. Recombine two parents

`crossover` receives two selected parents as fitness dictionaries and returns one new coefficient list. It randomly chooses two of the four positions from parent 1 and takes the other two from parent 2, preserving the original feature order.

For example, if positions 0 and 2 are chosen from parent A:

```text
Parent A: [a1, a2, a3, a4]
Parent B: [b1, b2, b3, b4]
Child:    [a1, b2, a3, b4]
```

This is not the fixed midpoint split used in the previous notebook. Nor does crossover average coefficients: it copies their values. The positions chosen can differ for every child.

**Pause and predict:** Can two individually good parents produce a child with higher error than either parent? Think about how several coefficients can compensate for one another in a prediction.

In [45]:
def crossover(parent_1, parent_2):
    child = {}
    loci = [i for i in range(0, individual_size)]
    loci_1 = sample(loci, floor(0.5*(individual_size)))
    loci_2 = [i for i in loci if i not in loci_1]
    chromosome_1 = [[i, parent_1['coeff'][i]] for i in loci_1]
    chromosome_2 = [[i, parent_2['coeff'][i]] for i in loci_2]
    child.update({key: value for (key, value) in chromosome_1})
    child.update({key: value for (key, value) in chromosome_2})
    return [child[i] for i in loci]

## 9. Mutate selected children

`mutate` chooses gene positions without replacement and adds a random signed change to each selected coefficient. The sign is equally likely to be positive or negative, and the magnitude is drawn from `[0, 1)`. No clipping is applied, so coefficients can become negative or exceed one.

The parameter name `probability_of_gene_mutating` is potentially misleading. The function uses it to compute a **fixed count**:

$$k=\lfloor p_{\mathrm{gene}}\,d\rfloor,$$

where $d$ is the number of genes. With four genes and `p_gene = 0.25`, exactly **one gene** is selected per mutated child. There is no separate probability trial for each gene. With four genes, values below 0.25 select zero genes.

The function changes the supplied list in place. Mutation can improve or worsen fitness; subsequent selection decides which candidates reproduce. The possible changes are large relative to the true coefficients, so fine adjustment near the optimum can be slow.

In [46]:
def mutate(individual):
    loci = [i for i in range(0, individual_size)]
    no_of_genes_mutated = floor(probability_of_gene_mutating*individual_size)
    loci_to_mutate = sample(loci, no_of_genes_mutated)
    for locus in loci_to_mutate:
        gene_transform = choice([-1, 1])
        change = gene_transform*random()
        individual[locus] = individual[locus] + change
    return individual

## 10. Replace the population

`get_new_generation` performs these steps:

1. Draw 1,000 parent pairs from the selected pool. The two entries in each pair are distinct, but parents may appear in many different pairs.
2. Create one child from each pair using crossover.
3. Select exactly $\lfloor p_{\mathrm{individual}}N\rfloor$ children without replacement for mutation.
4. Return the offspring as the entire next population.

At the defaults, exactly **100 of the 1,000 children** are selected for mutation, and each of those has one gene selected. As with gene mutation, `probability_of_individual_mutating` determines a count rather than independent probability trials.

There is **no explicit elitism**: the best parents are not copied unchanged into reserved slots. A strong candidate may survive through reproduction, but that is not guaranteed. Consequently, the best MSE may increase from one generation to the next, unlike the elitist example in `genetic_algorithm.ipynb`.

In [47]:
def get_new_generation(selected_individuals):
    parent_pairs = [sample(selected_individuals, 2)
                    for i in range(population_size)]
    offspring = [crossover(pair[0], pair[1]) for pair in parent_pairs]
    offspring_indices = [i for i in range(population_size)]
    offspring_to_mutate = sample(
        offspring_indices,
        floor(probability_of_individual_mutating*population_size)
    )
    mutated_offspring = [[i, mutate(offspring[i])]
                         for i in offspring_to_mutate]
    for child in mutated_offspring:
        offspring[child[0]] = child[1]
    return offspring

## 11. Run the search and trace the bookkeeping

The main cell generates a fresh dataset and computes `best_possible`, the least-squares reference for **that same dataset**. This reference does not guide reproduction; it is available for comparison after the run.

| Setting | Default | Effect |
| --- | --- | --- |
| `individual_size` | 4 | Coefficients in each candidate |
| `population_size` | 1,000 | Offspring generated each iteration |
| `selection_size` | 100 | Lowest-error candidates eligible to reproduce |
| `max_generations` | 100 | Counter limit; see the timing note below |
| `probability_of_individual_mutating` | 0.10 | Select 100 offspring for mutation |
| `probability_of_gene_mutating` | 0.25 | Select one gene in each mutated offspring |

If you change population size, also replace the literal `1000` in `create_population(individual_size, 1000)` with the intended size. Keep at least two selected parents, since each mating samples two distinct entries. Keep mutation fractions between zero and one.

### What happens in one loop iteration?

1. Score the latest entry already in `best_individuals_stash`.
2. Evaluate the current population and append its winner to the stash.
3. Generate a complete offspring population.
4. Check termination using the score from step 1, then increment the counter.

This order explains two surprises in the output. A generation reaching 99.9 can be followed by another evaluation before stopping; and reaching the counter limit of 100 permits 101 evaluations, beginning at zero. The stash starts with a separate random candidate, so the first stopping check does not use the initial population's winner.

The offspring created on the terminating iteration are **not evaluated**. The final printed result is the last evaluated generation's winner, not necessarily the best candidate across the run. With no elitism, it could even fall below the threshold that caused termination based on the previous winner.

These are limitations of the existing loop, not essential features of a GA. For this walkthrough, use the printed generation numbers and the stash history to distinguish the intended stopping rule from what the code actually evaluates.

In [48]:
# Generate initial population
inputs, outputs = generate_data()

# Set parameters
individual_size = len(inputs[0])
population_size = 1000
selection_size = floor(0.1*population_size)
max_generations = 100
probability_of_individual_mutating = 0.1
probability_of_gene_mutating = 0.25

# Run the genetic algorithm
best_possible = multiple_linear_regression(inputs, outputs)
best_individuals_stash = [create_individual(individual_size)]
initial_population = create_population(individual_size, 1000)
current_population = initial_population
termination = False
generation_count = 0
while termination is False:
    current_best_individual = get_fitness(best_individuals_stash[-1], inputs)
    print('Generation: ', generation_count)
    best_individuals = evaluate_population(current_population)
    current_population = get_new_generation(best_individuals)
    termination = check_termination_condition(current_best_individual)
    generation_count += 1
else:
    print(get_fitness(best_individuals_stash[-1], inputs))

Generation:  0


100%|██████████| 1000/1000 [00:00<00:00, 1486.25it/s]


Error:  0.043889053754070165 COD:  -80.30512364441604
Generation:  1


100%|██████████| 1000/1000 [00:00<00:00, 1530.29it/s]


Error:  0.01065633553500699 COD:  56.22161491559534
Generation:  2


...

Error:  2.970001406781938e-05 COD:  99.87798632573065
Generation:  61


100%|██████████| 1000/1000 [00:00<00:00, 1445.58it/s]


Error:  1.5370266380925453e-05 COD:  99.93685583207628
Generation:  62


100%|██████████| 1000/1000 [00:00<00:00, 1439.70it/s]

Error:  1.5370266380925453e-05 COD:  99.93685583207628
{'COD': 99.93685583207628, 'error': 1.5370266380925453e-05, 'coeff': [0.4047252278048127, -0.2979875966258534, 0.20430589547127287, -0.10493415026627206]}


## 12. Interpret the saved result

The saved run reports coefficients approximately

$$\hat{\boldsymbol{\beta}}=(0.4047,-0.2980,0.2043,-0.1049),$$

with MSE $\approx1.54\times10^{-5}$ and COD $\approx99.9369$ ($R^2\approx0.999369$). These are close to the known coefficients, but they are not exact. Your unseeded run will generally produce different numbers and may stop at the generation limit instead.

The least-squares solution gives essentially zero error for this noiseless linear model. The GA demonstrates that selection and variation can discover a good approximation; it does not outperform the direct solution here. Compare against `best_possible` from the main run, rather than assuming the earlier demonstration used the same observations.

All reported scores use the data used for selection. There is **no train/test split**, no measurement noise, and no uncertainty estimate. A high training COD alone does not establish generalization or reliable coefficient estimates for real experimental data. Correlated features, noise, and a missing intercept can all change the interpretation.

## 13. Explore and discuss

1. **Verify one prediction.** Use the known coefficients and the observation `(0.5, 0.2, 0.8, 0.1)` to obtain `0.29`. Explain why squaring residuals prevents positive and negative errors from cancelling.
2. **Score the known answer.** After generating a dataset, evaluate `get_fitness([0.4, -0.3, 0.2, -0.1], inputs)`. Expect essentially zero MSE and COD near 100. Compare with `best_possible`.
3. **Remove mutation.** Set the individual-mutation fraction to zero and rerun the main cell. Explain why crossover among initially nonnegative genes cannot recover the negative true coefficients, even if the score improves.
4. **Inspect mutation counts.** With four genes, predict the number changed for gene-mutation fractions 0.20, 0.25, and 0.50. Why do these settings select zero, one, and two positions rather than a variable count?
5. **Track progress honestly.** Score the vectors in `best_individuals_stash[1:]` on the current dataset, excluding the initial random placeholder. Plot generation MSE and its cumulative minimum. Why can the first curve rise while the second cannot? Do this before regenerating the data.
6. **Improve the loop as a code exercise.** Check the newly evaluated winner before creating offspring, define the limit as a count of evaluated populations, and preserve a separate best-so-far candidate. Optionally add elitism. Explain which change affects reporting, which affects stopping, and which changes the search itself.
7. **Make the regression more realistic.** Add controlled noise to the targets and evaluate both methods on held-out observations. Keep data and splits fixed when comparing seeds. If you add an intercept, include a column of ones in the design matrix and revisit the chromosome length and mutation count. Why should a noisy problem not be expected to reach COD = 100?

## What to take away

A regression model defines predictions; a loss function measures their mismatch with observations; an optimizer searches for coefficients that reduce that loss. The GA is one optimizer, while least squares provides a direct reference for this model. Understanding the data, the objective, and the exact implementation is as important as obtaining a high fitness score.